# Exoplot Backend Testing Notebook
This notebook tests every single module in the `modules/` directory. It demonstrates how to load data, apply chained filters, plot catalogs, process lightcurves, and run MCMC fits.

In [1]:
# 1. Imports and Setup
import pandas as pd
from IPython.display import display, HTML

# Import all our custom modules
from modules.constants import MCMC_BOUNDS, MODEL_CATALOG
from modules.models import MassRadiusModels
from modules.catalog import ExoplanetCatalog
from modules.plotting import CatalogPlotter, TransitPlotter
from modules.lightcurve import LightCurveAnalyzer
from modules.mcmc import TransitFitter

/Users/simon.wtmn/Desktop/Exoplot_ENS/exoplotvenv/lib/python3.13/site-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


## 1. Constants & Theoretical Models (`constants.py` & `models.py`)
Verify that the hardcoded constants are accessible and that we can load a theoretical mass-radius curve from the `theoretical_models/` folder.

In [2]:
# Instantiate the model loader
mr_models = MassRadiusModels()
print(mr_models.models_directory)

print('\n------------------------------\n')

# Methods presentation
print('All models :')
display(mr_models.list_available_models())

print('------------------------------\n')

print('Specific Label Name :')
display(mr_models.get_model_label('aguichine_2025'))

print('------------------------------\n')

print('Model Values for "Aguichine 2025" :')
display(mr_models.get_model_curve('aguichine_2025'))

/Users/simon.wtmn/Desktop/Exoplot_ENS/data/theoretical_models

------------------------------

All models :


{'zeng_rocky': ('zeng_2019_pure_rock', 'Zeng+2019: Pure Rock'),
 'zeng_iron': ('zeng_2019_pure_iron', 'Zeng+2019: Pure Iron'),
 'zeng_earth': ('zeng_2019_earth_like', 'Zeng+2019: Earth-like'),
 'zeng_2016_20fe': ('zeng_2016_20_Fe', 'Zeng+2016: 20% Iron'),
 'marcus_collision': ('marcus_2010_maximum_collision_stripping',
  'Marcus+2010: Collision'),
 'Water World': ('MR-Water20_650K_DORN.txt',
  'Water World: 650K',
  {'usecols': [0, 1], 'header': 0}),
 'luo_2024': ('luo_2024.txt', 'Luo 2024', {'usecols': [0, 1], 'header': 0}),
 'aguichine_2021': ('aguichine_2021',
  'Aguichine 2021',
  {'usecols': [4, 6], 'header': 0}),
 'aguichine_2025': ('aguichine_2025.dat',
  'Aguichine 2025',
  {'usecols': [3, 6], 'header': 0}),
 'tang_2025': ('tang_2025.dat', 'Tang 2025', {'usecols': [3, 6], 'header': 0}),
 'tang_2025_boiloff': ('tang_2025_boiloff.csv',
  'Tang 2025: Boil-off',
  {'usecols': [0, 1], 'header': 0, 'sep': ','}),
 'LF_100Myr_solar': ('Lopez&Fortney_2014_100Myr_solar',
  'L&F 2014: 100

------------------------------

Specific Label Name :


'Aguichine 2025'

------------------------------

Model Values for "Aguichine 2025" :


,mass,radius
0,0.2,1.18359
1,0.2,0.83241
2,0.2,0.81833
3,0.2,0.80755
4,0.2,0.79838
...,...,...
60463,20.0,3.41811
60464,20.0,3.41802
60465,20.0,3.41787
60466,20.0,3.41757


## 2. Exoplanet Catalog Filtering (`catalog.py`)
Instantiate the catalog using the `NEA.csv` file. 

In [3]:
# Instantiate the catalog loader
catalog = ExoplanetCatalog(dataset_name='NEA')
print(f"Original dataset size: {len(catalog.original_df)} planets")

catalog.filter_discovery(mission='Kepler') \
       .filter_spectral_type('M') \
       .filter_planet(mass_max=15)

filtered_df = catalog.get_data()
print(f"Filtered dataset size: {len(catalog.get_data())} planets")
display(catalog.get_data()[['pl_name', 'st_spectype', 'pl_bmasse', 'pl_rade']].head())

catalog.reset()
print(f"Dataset size after reset: {len(catalog.original_df)} planets")

Original dataset size: 6128 planets
Filtered dataset size: 109 planets


,pl_name,st_spectype,pl_bmasse,pl_rade
0,Kepler-1089 b,NaN,4.01,1.83
1,Kepler-1124 b,NaN,2.18,1.28
2,Kepler-114 b,M0 V,1.09,1.26
3,Kepler-114 c,M0 V,2.80,1.60
4,Kepler-114 d,M0 V,3.90,2.53


Dataset size after reset: 6128 planets


## 3. Catalog Visualizations (`plotting.py`)
Use the `CatalogPlotter` to visualize the filtered dataset. Plot Mass vs. Radius and overlay the theoretical Earth-like and Pure Iron models.

In [4]:
# Instantiate the catalog plotter
cat_plotter = CatalogPlotter()

print("Generating 1. Basic Scatter Plot (Mass vs. Radius with Models)...")
scatter_html = cat_plotter.plot_scatter(
    df=filtered_df,
    x_col='pl_bmasse',
    y_col='pl_rade',
    log_x=True,
    log_y=False,
    highlight_planets=['Kepler-10 b'],
    overlay_models=['zeng_earth', 'zeng_iron', 'Water World'] 
)
display(HTML(scatter_html))

print("Generating 2. Colored Scatter Plot (Orbital Period vs. Eccentricity)...")
colored_scatter_html = cat_plotter.plot_scatter(
    df=filtered_df,
    x_col='pl_orbper',
    y_col='pl_orbeccen',
    color_by='st_teff',
    log_x=True,
    log_y=False
)
display(HTML(colored_scatter_html))

print("Generating 3. 2D Density Heatmap (Mass vs. Density)...")
density_html = cat_plotter.plot_density(
    df=filtered_df,
    x_col='pl_bmasse',
    y_col='pl_dens',
    log_x=True,
    log_y=False,
    cmap='Viridis'
)
display(HTML(density_html))

print("Generating 4. 1D Histogram (Distribution of Planetary Radii)...")
histogram_html = cat_plotter.plot_histogram(
    df=filtered_df,
    column='pl_rade',
    bins=100,
    log_x=False,
    log_y=False,
    color='orange'
)
display(HTML(histogram_html))


Generating 1. Basic Scatter Plot (Mass vs. Radius with Models)...


Generating 2. Colored Scatter Plot (Orbital Period vs. Eccentricity)...


Generating 3. 2D Density Heatmap (Mass vs. Density)...


Generating 4. 1D Histogram (Distribution of Planetary Radii)...


## 4. Lightcurve Processing (`lightcurve.py`)
Using `LightCurveAnalyzer` to fetch data from MAST (using Lightkurve), clean it, find the period via BLS, and fold it.

In [5]:
# Instantiate the analyzer
analyzer = LightCurveAnalyzer("WASP 76")

# 1. Search MAST archive (fetching metadata)
print("Searching MAST...")
search_results = analyzer.search()
display(search_results)

# 2. Download and clean a result
print("\nDownloading and cleaning data...")
analyzer.download_and_clean(index=0)

print("-----------------------------------")

# 3. Compute the BLS periodogram
print("Computing BLS periodogram...")
analyzer.compute_periodogram()
print(f"Best Period Found: {analyzer.best_period:.5f} days")

print("-----------------------------------")

# 4. Fold the lightcurve
print("Folding lightcurve...")
analyzer.fold_lightcurve(harmonic=1)
print("Lightcurve successfully processed")

Searching MAST...


,mission,year,author,exptime,target_name,distance
0,TESS Sector 30,2020,SPOC,120.0,293435336,0.0
1,TESS Sector 42,2021,SPOC,120.0,293435336,0.0
2,TESS Sector 43,2021,SPOC,120.0,293435336,0.0
3,TESS Sector 97,2025,SPOC,20.0,293435336,0.0
4,TESS Sector 97,2025,SPOC,120.0,293435336,0.0
5,TESS Sector 30,2020,TESS-SPOC,600.0,293435336,0.0
6,TESS Sector 42,2021,TESS-SPOC,600.0,293435336,0.0
7,TESS Sector 43,2021,TESS-SPOC,600.0,293435336,0.0
8,TESS Sector 30,2020,QLP,600.0,293435336,0.0
9,TESS Sector 42,2021,QLP,600.0,293435336,0.0



-----------------------------------
Computing BLS periodogram...
Best Period Found: 1.81238 days
-----------------------------------
Folding lightcurve...
Lightcurve successfully processed


## 5. Transit Visualizations (`plotting.py`)
Using the `TransitPlotter` to visualize the BLS periodogram and the folded lightcurve generated by our analyzer.

In [6]:
# Plot the Periodogram
periodogram_html = TransitPlotter.plot_periodogram(
    x=analyzer.periodogram.period.value,
    y=analyzer.periodogram.power.value,
    title=f"BLS Periodogram for {analyzer.target_name}",
    xaxis_type='period'
)
display(HTML(periodogram_html))

# Plot the Folded Lightcurve
lc_html = TransitPlotter.plot_lightcurve(
    x=analyzer.folded_lc.time.jd,
    y=analyzer.folded_lc.flux.value,
    err=analyzer.folded_lc.flux_err.value if analyzer.folded_lc.flux_err is not None else None,
    title=f"Folded Lightcurve ({analyzer.target_name})",
    style='scatter',
    xlabel="Time (Phase)",
    ylabel="Normalized Flux"
)
display(HTML(lc_html))

## 6. MCMC Transit Fitting (`mcmc.py`)
Extracting the cleaned numpy arrays from our analyzer and pass them to the `TransitFitter`.

In [9]:
from tqdm.notebook import tqdm

time, flux, err, period = analyzer.get_mcmc_data()

# 
# 1. User defines custom bounds from the UI
custom_bounds = [
    (0.001, 0.2),      # Minimum and maximum planetary radius ratio
    (89.6, 95),        # Inclination angle in degrees
    (4, 4.2),          # Semi-major axis to stellar radius ratio
    (-0.04, 0.04),     # Mid-transit time offset limits
]

custom_x0 = [0.02, 85.0, 10.0, 0.0]

# Instantiate fitter WITH the custom constraints
fitter = TransitFitter(time=time, flux=flux, flux_err=err, period=period,
                       custom_bounds=custom_bounds, 
                       custom_x0=custom_x0
)

# Parameters for our test (Increased walkers and steps because multiprocessing is fast!)
n_walkers = 256
n_steps = 20000

print("Starting Parallel MCMC Optimization...")

pbar = tqdm(total=n_steps, desc="Sampling Posterior", unit="steps")
last_step = [0]

def update_pbar(current_step, total_steps):
    increment = current_step - last_step[0]
    pbar.update(increment)
    last_step[0] = current_step

# ==========================================
# 2. RUN WITH MULTIPROCESSING
# ==========================================
results = fitter.run_mcmc(
    nwalkers=n_walkers, 
    nsteps=n_steps, 
    progress_callback=update_pbar,
    use_multiprocessing=True # Uses all available cores on your Mac!
)

pbar.update(n_steps - last_step[0])
pbar.close()

print("\nMCMC Complete! Results:")
for param, (median, upper, lower) in results.items():
    print(f"{param}: {median:.5f} (+{upper:.5f} / -{lower:.5f})")

Starting Parallel MCMC Optimization...


Sampling Posterior:   0%|          | 0/20000 [00:00<?, ?steps/s]

emcee: Exception while calling your likelihood function:
  params: [1.06192744e-01 9.44046763e+01 4.00028953e+00 8.50747581e-03]
  args: []
  kwargs: {}
  exception:


Process SpawnPoolWorker-24:
Process SpawnPoolWorker-23:
Process SpawnPoolWorker-22:
Process SpawnPoolWorker-21:
Process SpawnPoolWorker-20:
Process SpawnPoolWorker-13:
Process SpawnPoolWorker-17:
Process SpawnPoolWorker-14:
Process SpawnPoolWorker-19:
Process SpawnPoolWorker-18:
Process SpawnPoolWorker-15:
Process SpawnPoolWorker-16:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessi

KeyboardInterrupt: 

## 7. MCMC Diagnostics and Final Fit
Visualizing the MCMC results. We will generate the trace plots (to check convergence), the corner plot (to check parameter covariance), and overlay the best-fit Batman model over our original lightcurve data.

In [8]:
# 1. Trace Plots
trace_html = TransitPlotter.plot_mcmc_traces(
    flat_samples=fitter.flat_samples,
    labels=list(results.keys()),
    theme='dark'
)
display(HTML(trace_html))

# 2. Corner Plot
corner_html = TransitPlotter.plot_mcmc_corner(
    flat_samples=fitter.flat_samples,
    labels=list(results.keys()),
    theme='dark'
)
display(HTML(corner_html))

# 3. Final Model Overlay
model_time, model_flux = fitter.get_best_model_curve()

final_fit_html = TransitPlotter.plot_lightcurve(
    x=time, y=flux, err=err,
    model_x=model_time, model_y=model_flux,
    title=f"MCMC Best Fit: {analyzer.target_name}",
    style='scatter',
    bins=50,
    xlabel="Time (Phase)",
    ylabel="Normalized Flux",
    theme='dark'
)
display(HTML(final_fit_html))